In [ ]:
# Repository bootstrap — portable local/Jupyter execution
from pathlib import Path
import os, sys

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "koa_severity").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
from koa_severity.data import get_dataset_path
DATASET_PATH = get_dataset_path()
OUTPUTS_DIR = PROJECT_ROOT / "results"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"
OUTPUTS_DIR.mkdir(exist_ok=True)
CHECKPOINTS_DIR.mkdir(exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)


In [ ]:
# ============================================================
# CELL 1 — Environment Setup
# ============================================================

import os
import random
import numpy as np
import torch

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("ENVIRONMENT CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 2 — Dataset Setup
# ============================================================

import os

# ------------------------------------------------------------
# Download / load dataset
# ------------------------------------------------------------

DATASET_PATH = get_dataset_path()

TRAIN_DIR = os.path.join(
    DATASET_PATH,
    "train"
)

VAL_DIR = os.path.join(
    DATASET_PATH,
    "val"
)

TEST_DIR = os.path.join(
    DATASET_PATH,
    "test"
)

AUTO_TEST_DIR = os.path.join(
    DATASET_PATH,
    "auto_test"
)

print("=" * 70)
print("DATASET READY")
print("=" * 70)

print("Dataset path:")
print(DATASET_PATH)

print("\nTrain:", os.path.exists(TRAIN_DIR))
print("Val:", os.path.exists(VAL_DIR))
print("Test:", os.path.exists(TEST_DIR))
print("Auto test:", os.path.exists(AUTO_TEST_DIR))

print("=" * 70)

In [ ]:
# ============================================================
# CELL 3 — Best Data Pipeline
# ============================================================

import cv2
import numpy as np
import torch

from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler


# ------------------------------------------------------------
# 1. CLAHE transform
# ------------------------------------------------------------

class CLAHETransform:

    def __init__(
        self,
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):

        img = np.array(img)

        if len(img.shape) == 3:
            img = cv2.cvtColor(
                img,
                cv2.COLOR_RGB2GRAY
            )

        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit,
            tileGridSize=self.tile_grid_size
        )

        img = clahe.apply(img)

        return Image.fromarray(img)


# ------------------------------------------------------------
# 2. Training transform
# ------------------------------------------------------------

train_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=5
    ),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05)
    ),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# 3. Validation / Test transform
# ------------------------------------------------------------

eval_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# 4. Datasets
# ------------------------------------------------------------

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=eval_transform
)


# ------------------------------------------------------------
# 5. WeightedRandomSampler
# ------------------------------------------------------------

train_targets = np.array(
    train_dataset.targets
)

class_counts = np.bincount(
    train_targets,
    minlength=5
)

class_sample_weights = (
    1.0 / class_counts
)

sample_weights = class_sample_weights[
    train_targets
]

sample_weights = torch.DoubleTensor(
    sample_weights
)

weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)


# ------------------------------------------------------------
# 6. DataLoaders
# ------------------------------------------------------------

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=weighted_sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# 7. Check
# ------------------------------------------------------------

images, labels = next(
    iter(train_loader)
)

print("=" * 70)
print("BEST DATA PIPELINE READY")
print("=" * 70)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

print("\nClass mapping:")
print(train_dataset.class_to_idx)

print("\nClass counts:")
for i, count in enumerate(class_counts):
    print(f"KL{i}: {count}")

print("\nBatch shape:")
print(images.shape)

print("\nDevice:")
print(device)

print("\nGPU:")
print(torch.cuda.get_device_name(0))

print("=" * 70)

In [ ]:
# ============================================================
# CELL 4 — Build Winning ResNet18 Baseline
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models


# ------------------------------------------------------------
# Create pretrained ResNet18
# ------------------------------------------------------------

model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model.fc = nn.Linear(
    model.fc.in_features,
    5
)

model = model.to(device)


# ------------------------------------------------------------
# Stage 1:
# Freeze backbone, train classifier head only
# ------------------------------------------------------------

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Winning loss:
# Unweighted CrossEntropy
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("=" * 70)
print("WINNING RESNET18 — READY")
print("=" * 70)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print("Input size           : 224x224")
print("Preprocessing        : CLAHE + augmentation + ImageNet normalization")
print("Sampler              : WeightedRandomSampler")
print("Loss                 : Unweighted CrossEntropy")
print("Learning rate        : 1e-4")
print("Backbone             : Frozen")
print("Device               :", device)
print("GPU                  :", torch.cuda.get_device_name(0))

print("=" * 70)

In [ ]:
# ============================================================
# CELL 5 — Train Winning ResNet18
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 5

best_macro_f1 = -1.0
best_epoch = 0

BEST_MODEL_PATH = "./best_resnet18_winner.pth"

print("=" * 70)
print("TRAINING WINNING RESNET18")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = (
        sum(
            p == y
            for p, y in zip(
                all_predictions,
                all_labels
            )
        )
        / val_total
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # SAVE BEST CHECKPOINT
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1:

        best_macro_f1 = macro_f1
        best_epoch = epoch + 1

        torch.save(
            {
                "epoch": best_epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc
            },
            BEST_MODEL_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Best Macro F1: {best_macro_f1:.4f}"
)

print(
    f"Best epoch: {best_epoch}"
)

print(
    f"Saved checkpoint: {BEST_MODEL_PATH}"
)

print("=" * 70)

In [ ]:
# ============================================================
# EXPERIMENT D — CORRECT MODEL SETUP
# CLAHE + Augmentation + ImageNet Normalization
# Full ResNet18 Fine-Tuning
# Weighted CrossEntropy
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models


# ------------------------------------------------------------
# 1. Calculate class weights from TRAINING data
# ------------------------------------------------------------

train_class_counts = torch.tensor(
    class_counts,
    dtype=torch.float32
)

class_weights = (
    len(train_dataset)
    /
    (
        5 *
        train_class_counts
    )
)

class_weights_tensor = class_weights.to(
    device
)


# ------------------------------------------------------------
# 2. Build fresh pretrained ResNet18
# ------------------------------------------------------------

model_D = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model_D.fc = nn.Linear(
    model_D.fc.in_features,
    5
)

model_D = model_D.to(device)


# ------------------------------------------------------------
# 3. Full fine-tuning
# ------------------------------------------------------------

for param in model_D.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# 4. Weighted CrossEntropy
# ------------------------------------------------------------

criterion_D = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ------------------------------------------------------------
# 5. Optimizer
# ------------------------------------------------------------

optimizer_D = optim.Adam(
    model_D.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# 6. Check
# ------------------------------------------------------------

total_params_D = sum(
    p.numel()
    for p in model_D.parameters()
)

trainable_params_D = sum(
    p.numel()
    for p in model_D.parameters()
    if p.requires_grad
)

print("=" * 70)
print("EXPERIMENT D — MODEL READY")
print("=" * 70)

print(
    f"Total parameters     : {total_params_D:,}"
)

print(
    f"Trainable parameters : {trainable_params_D:,}"
)

print("\nClass weights:")

for i, weight in enumerate(
    class_weights_tensor.cpu().numpy()
):
    print(
        f"KL{i}: {weight:.4f}"
    )

print("\nInput size           : 224x224")
print(
    "Preprocessing        : "
    "CLAHE + augmentation + ImageNet normalization"
)
print("Sampler              : Standard shuffled training")
print("Loss                 : Weighted CrossEntropy")
print("Learning rate        : 1e-4")
print("Backbone             : Fully trainable")
print("Device               :", device)
print(
    "GPU                  :",
    torch.cuda.get_device_name(0)
)

print("=" * 70)

In [ ]:
# ============================================================
# EXPERIMENT D — TRAINING
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 10

best_macro_f1_D = -1.0
best_epoch_D = 0

BEST_D_PATH = "./best_resnet18_oa_clahe_REBUILT.pth"

print("=" * 70)
print("EXPERIMENT D — TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model_D.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_D.zero_grad()

        outputs = model_D(images)

        loss = criterion_D(
            outputs,
            labels
        )

        loss.backward()

        optimizer_D.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = (
        train_correct /
        train_total
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model_D.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_D(images)

            loss = criterion_D(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = (
        sum(
            p == y
            for p, y in zip(
                all_predictions,
                all_labels
            )
        )
        / val_total
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # SAVE BEST CHECKPOINT
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_D:

        best_macro_f1_D = macro_f1
        best_epoch_D = epoch + 1

        torch.save(
            {
                "epoch": best_epoch_D,
                "model_state_dict":
                    model_D.state_dict(),
                "optimizer_state_dict":
                    optimizer_D.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
                "class_weights":
                    class_weights.tolist(),
                "experiment":
                    "D_rebuilt"
            },
            BEST_D_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )

print("-" * 70)

print(
    f"Best Macro F1: "
    f"{best_macro_f1_D:.4f}"
)

print(
    f"Best epoch: "
    f"{best_epoch_D}"
)

print(
    f"Checkpoint: "
    f"{BEST_D_PATH}"
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL — Reproducibility Setup
# ============================================================

import random
import numpy as np
import torch

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("=" * 70)
print("REPRODUCIBILITY SETUP")
print("=" * 70)

print("Seed:", SEED)
print("CUDA deterministic:", torch.backends.cudnn.deterministic)
print("CUDA benchmark:", torch.backends.cudnn.benchmark)

print("=" * 70)

In [ ]:
# ============================================================
# Reproducible DataLoader Setup
# ============================================================

def seed_worker(worker_id):
    worker_seed = (
        torch.initial_seed()
        % 2**32
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


# Generator for reproducible DataLoader shuffling
g = torch.Generator()
g.manual_seed(SEED)


print("=" * 70)
print("REPRODUCIBLE DATALOADER SETTINGS READY")
print("=" * 70)

print("Seed:", SEED)
print("Worker seeding: Enabled")
print("DataLoader generator seed:", SEED)

print("=" * 70)

In [ ]:
# ============================================================
# Reproducible DataLoaders for Experiment D
# ============================================================

from torch.utils.data import DataLoader

train_loader_repro = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=g
)

val_loader_repro = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker
)

test_loader_repro = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker
)

print("=" * 70)
print("REPRODUCIBLE DATALOADERS READY")
print("=" * 70)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

print("Batch size:", train_loader_repro.batch_size)
print("Workers:", train_loader_repro.num_workers)
print("Shuffle train:", train_loader_repro.shuffle if hasattr(train_loader_repro, "shuffle") else True)

print("=" * 70)

In [ ]:
# ============================================================
# Experiment D — Reproducible Model Setup
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models


# ------------------------------------------------------------
# Fresh pretrained ResNet18
# ------------------------------------------------------------

model_D_repro = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model_D_repro.fc = nn.Linear(
    model_D_repro.fc.in_features,
    5
)

model_D_repro = model_D_repro.to(device)


# ------------------------------------------------------------
# Full fine-tuning
# ------------------------------------------------------------

for param in model_D_repro.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Class weights
# ------------------------------------------------------------

class_counts_repro = np.bincount(
    np.array(train_dataset.targets),
    minlength=5
)

class_weights_repro = (
    len(train_dataset)
    /
    (
        5 * class_counts_repro
    )
)

class_weights_tensor_repro = torch.tensor(
    class_weights_repro,
    dtype=torch.float32,
    device=device
)


# ------------------------------------------------------------
# Weighted CrossEntropy
# ------------------------------------------------------------

criterion_D_repro = nn.CrossEntropyLoss(
    weight=class_weights_tensor_repro
)


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer_D_repro = optim.Adam(
    model_D_repro.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model_D_repro.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model_D_repro.parameters()
    if p.requires_grad
)

print("=" * 70)
print("EXPERIMENT D — REPRODUCIBLE MODEL READY")
print("=" * 70)

print("Seed:", SEED)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

print("\nClass weights:")
for i, w in enumerate(class_weights_repro):
    print(f"KL{i}: {w:.4f}")

print("\nLoss: Weighted CrossEntropy")
print("Learning rate: 1e-4")
print("Backbone: Fully trainable")
print("Resolution: 224x224")
print("Sampler: Standard shuffled loader")
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

print("=" * 70)

In [ ]:
# ============================================================
# Experiment D — Reproducible Training
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 10

best_macro_f1_D_repro = -1.0
best_epoch_D_repro = 0

BEST_D_REPRO_PATH = (
    "./best_resnet18_D_seed2026.pth"
)

print("=" * 70)
print("EXPERIMENT D — REPRODUCIBLE TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model_D_repro.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_repro:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_D_repro.zero_grad()

        outputs = model_D_repro(images)

        loss = criterion_D_repro(
            outputs,
            labels
        )

        loss.backward()

        optimizer_D_repro.step()

        train_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = (
        train_correct /
        train_total
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model_D_repro.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_repro:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_D_repro(images)

            loss = criterion_D_repro(
                outputs,
                labels
            )

            val_loss += (
                loss.item()
                * images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = (
        sum(
            p == y
            for p, y in zip(
                all_predictions,
                all_labels
            )
        )
        / val_total
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # SAVE BEST
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_D_repro:

        best_macro_f1_D_repro = macro_f1
        best_epoch_D_repro = epoch + 1

        torch.save(
            {
                "seed": SEED,
                "epoch": best_epoch_D_repro,
                "model_state_dict":
                    model_D_repro.state_dict(),
                "optimizer_state_dict":
                    optimizer_D_repro.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
                "class_weights":
                    class_weights_repro.tolist(),
                "resolution": "224x224",
                "preprocessing": "CLAHE",
                "loss": "WeightedCrossEntropy",
                "sampler": "standard_shuffle"
            },
            BEST_D_REPRO_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Seed: {SEED}"
)

print(
    f"Best Macro F1: "
    f"{best_macro_f1_D_repro:.4f}"
)

print(
    f"Best epoch: "
    f"{best_epoch_D_repro}"
)

print(
    f"Checkpoint: "
    f"{BEST_D_REPRO_PATH}"
)

print("=" * 70)

In [ ]:

print("Artifact saved to:", "./best_resnet18_D_seed2026.pth"
)

In [ ]:
# ============================================================
# SAMPLER EXPERIMENT — SEED 2026
# Load D checkpoint + WeightedRandomSampler
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from torch.utils.data import DataLoader, WeightedRandomSampler


# ------------------------------------------------------------
# 1. Load D checkpoint
# ------------------------------------------------------------

D_CHECKPOINT_PATH = (
    "./best_resnet18_D_seed2026.pth"
)

checkpoint_D = torch.load(
    D_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)


# ------------------------------------------------------------
# 2. Build fresh ResNet18
# ------------------------------------------------------------

model_sampler_2026 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model_sampler_2026.fc = nn.Linear(
    model_sampler_2026.fc.in_features,
    5
)

model_sampler_2026 = model_sampler_2026.to(device)


# ------------------------------------------------------------
# 3. Load trained D weights
# ------------------------------------------------------------

model_sampler_2026.load_state_dict(
    checkpoint_D["model_state_dict"]
)


# ------------------------------------------------------------
# 4. Stage 1: freeze backbone
# ------------------------------------------------------------

for param in model_sampler_2026.parameters():
    param.requires_grad = False

for param in model_sampler_2026.fc.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# 5. Build WeightedRandomSampler
# ------------------------------------------------------------

train_targets = np.array(
    train_dataset.targets
)

class_counts_sampler_2026 = np.bincount(
    train_targets,
    minlength=5
)

class_sampling_weights_2026 = (
    1.0 / class_counts_sampler_2026
)

sample_weights_2026 = (
    class_sampling_weights_2026[
        train_targets
    ]
)

sample_weights_2026 = torch.DoubleTensor(
    sample_weights_2026
)

weighted_sampler_2026 = WeightedRandomSampler(
    weights=sample_weights_2026,
    num_samples=len(sample_weights_2026),
    replacement=True
)


# ------------------------------------------------------------
# 6. New train loader
# ------------------------------------------------------------

train_loader_sampler_2026 = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=weighted_sampler_2026,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# 7. Keep validation unchanged
# ------------------------------------------------------------

val_loader_sampler_2026 = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# 8. Unweighted CrossEntropy
# ------------------------------------------------------------

criterion_sampler_2026 = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# 9. Optimizer
# ------------------------------------------------------------

optimizer_sampler_2026 = optim.Adam(
    model_sampler_2026.fc.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# 10. Check
# ------------------------------------------------------------

print("=" * 70)
print("SAMPLER EXPERIMENT — READY")
print("=" * 70)

print(
    f"Starting D Macro F1: "
    f"{checkpoint_D['macro_f1']:.4f}"
)

print(
    f"Starting D Val Acc: "
    f"{checkpoint_D['val_acc']:.4f}"
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model_sampler_2026.parameters()
        if p.requires_grad
    )
)

print(
    "Train samples:",
    len(train_dataset)
)

print(
    "Sampler: WeightedRandomSampler"
)

print(
    "Loss: Unweighted CrossEntropy"
)

print(
    "Learning rate: 1e-4"
)

print("=" * 70)

In [ ]:
# ============================================================
# SAMPLER EXPERIMENT — SEED 2026
# Stage 1 Training
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 5

best_macro_f1_sampler_2026 = checkpoint_D["macro_f1"]
best_epoch_sampler_2026 = checkpoint_D["epoch"]

BEST_SAMPLER_2026_PATH = (
    "./best_resnet18_sampler_seed2026.pth"
)

print("=" * 70)
print("SAMPLER EXPERIMENT — TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model_sampler_2026.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_sampler_2026:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_sampler_2026.zero_grad()

        outputs = model_sampler_2026(images)

        loss = criterion_sampler_2026(
            outputs,
            labels
        )

        loss.backward()

        optimizer_sampler_2026.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model_sampler_2026.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_sampler_2026:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_sampler_2026(images)

            loss = criterion_sampler_2026(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = (
        sum(
            p == y
            for p, y in zip(
                all_predictions,
                all_labels
            )
        )
        / val_total
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # SAVE BEST
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_sampler_2026:

        best_macro_f1_sampler_2026 = macro_f1

        best_epoch_sampler_2026 = (
            checkpoint_D["epoch"]
            + epoch
            + 1
        )

        torch.save(
            {
                "seed": SEED,
                "epoch": best_epoch_sampler_2026,
                "model_state_dict":
                    model_sampler_2026.state_dict(),
                "optimizer_state_dict":
                    optimizer_sampler_2026.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
                "starting_D_macro_f1":
                    checkpoint_D["macro_f1"],
                "sampler":
                    "WeightedRandomSampler",
                "loss":
                    "UnweightedCrossEntropy"
            },
            BEST_SAMPLER_2026_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Starting D Macro F1: "
    f"{checkpoint_D['macro_f1']:.4f}"
)

print(
    f"Best Sampler Macro F1: "
    f"{best_macro_f1_sampler_2026:.4f}"
)

print(
    f"Best epoch: "
    f"{best_epoch_sampler_2026}"
)

print(
    f"Checkpoint: "
    f"{BEST_SAMPLER_2026_PATH}"
)

print("=" * 70)

In [ ]:

print("Artifact saved to:", "./best_resnet18_sampler_seed2026.pth"
)

In [ ]:
# ============================================================
# Evaluate Seed 2026 + WeightedRandomSampler Model
# ============================================================

import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    cohen_kappa_score,
    classification_report,
    confusion_matrix
)


# ------------------------------------------------------------
# Load best Sampler checkpoint
# ------------------------------------------------------------

SAMPLER_CHECKPOINT_PATH = (
    "./best_resnet18_sampler_seed2026.pth"
)

checkpoint_sampler_final = torch.load(
    SAMPLER_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model_sampler_2026.load_state_dict(
    checkpoint_sampler_final["model_state_dict"]
)

model_sampler_2026 = model_sampler_2026.to(device)
model_sampler_2026.eval()


# ------------------------------------------------------------
# Validation predictions
# ------------------------------------------------------------

all_labels_final = []
all_predictions_final = []

with torch.no_grad():

    for images, labels in val_loader_sampler_2026:

        images = images.to(
            device,
            non_blocking=True
        )

        outputs = model_sampler_2026(
            images
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_labels_final.extend(
            labels.cpu().numpy()
        )

        all_predictions_final.extend(
            predictions.cpu().numpy()
        )


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

accuracy_final = accuracy_score(
    all_labels_final,
    all_predictions_final
)

balanced_accuracy_final = (
    balanced_accuracy_score(
        all_labels_final,
        all_predictions_final
    )
)

macro_f1_final = f1_score(
    all_labels_final,
    all_predictions_final,
    average="macro",
    zero_division=0
)

mae_final = mean_absolute_error(
    all_labels_final,
    all_predictions_final
)

qwk_final = cohen_kappa_score(
    all_labels_final,
    all_predictions_final,
    weights="quadratic"
)


# ------------------------------------------------------------
# Print overall results
# ------------------------------------------------------------

print("=" * 70)
print("SEED 2026 + SAMPLER — VALIDATION RESULTS")
print("=" * 70)

print(
    f"Checkpoint epoch   : "
    f"{checkpoint_sampler_final['epoch']}"
)

print(
    f"Saved Macro F1     : "
    f"{checkpoint_sampler_final['macro_f1']:.4f}"
)

print("\nOverall metrics:")

print(
    f"Accuracy           : "
    f"{accuracy_final:.4f}"
)

print(
    f"Balanced Accuracy  : "
    f"{balanced_accuracy_final:.4f}"
)

print(
    f"Macro F1           : "
    f"{macro_f1_final:.4f}"
)

print(
    f"MAE                : "
    f"{mae_final:.4f}"
)

print(
    f"QWK                : "
    f"{qwk_final:.4f}"
)


# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

class_names = [
    "Normal",
    "Doubtful",
    "Mild",
    "Moderate",
    "Severe"
]

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        all_labels_final,
        all_predictions_final,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm_final = confusion_matrix(
    all_labels_final,
    all_predictions_final
)

print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print("Rows = True class")
print("Cols = Predicted class\n")

print(
    "             " +
    "  ".join(
        f"{name[:9]:>9}"
        for name in class_names
    )
)

for i, row in enumerate(cm_final):

    print(
        f"{class_names[i]:<10}" +
        "  ".join(
            f"{value:9d}"
            for value in row
        )
    )

print("=" * 70)

In [ ]:
# ============================================================
# HYBRID EXPERIMENT
# Step 1 — Extract 512-D Deep Features
# From Seed 2026 + Sampler Checkpoint
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from tqdm.auto import tqdm
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader


# ------------------------------------------------------------
# 1. Load handcrafted feature tables
# ------------------------------------------------------------

train_roi_df = pd.read_csv(
    "./train_roi_classical_features.csv"
)

val_roi_df = pd.read_csv(
    "./val_roi_classical_features.csv"
)


# ------------------------------------------------------------
# 2. Evaluation preprocessing
#    EXACTLY the same preprocessing used by CNN
# ------------------------------------------------------------

deep_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# 3. Dataset using image_path from saved feature tables
# ------------------------------------------------------------

class DeepFeatureDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        image_path = self.df.loc[
            idx,
            "image_path"
        ]

        label = int(
            self.df.loc[
                idx,
                "label"
            ]
        )

        image = Image.open(
            image_path
        ).convert("L")

        image = self.transform(
            image
        )

        return image, label


# ------------------------------------------------------------
# 4. Build datasets/loaders
# ------------------------------------------------------------

train_deep_dataset = DeepFeatureDataset(
    train_roi_df,
    deep_transform
)

val_deep_dataset = DeepFeatureDataset(
    val_roi_df,
    deep_transform
)

train_deep_loader = DataLoader(
    train_deep_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

val_deep_loader = DataLoader(
    val_deep_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# 5. Build ResNet18
# ------------------------------------------------------------

model_deep = models.resnet18(
    weights=None
)

model_deep.fc = nn.Linear(
    model_deep.fc.in_features,
    5
)

model_deep = model_deep.to(device)


# ------------------------------------------------------------
# 6. Load BEST Seed 2026 + Sampler checkpoint
# ------------------------------------------------------------

SAMPLER_CHECKPOINT = (
    "./best_resnet18_sampler_seed2026.pth"
)

checkpoint_deep = torch.load(
    SAMPLER_CHECKPOINT,
    map_location=device,
    weights_only=False
)

model_deep.load_state_dict(
    checkpoint_deep["model_state_dict"]
)


# ------------------------------------------------------------
# 7. Remove classification head
#    Output becomes 512-D ResNet representation
# ------------------------------------------------------------

model_deep.fc = nn.Identity()

model_deep.eval()


# ------------------------------------------------------------
# 8. Extract features
# ------------------------------------------------------------

def extract_deep_features(
    loader,
    split_name
):

    features = []
    labels = []

    with torch.no_grad():

        for images, batch_labels in tqdm(
            loader,
            desc=f"Extracting {split_name}"
        ):

            images = images.to(
                device,
                non_blocking=True
            )

            outputs = model_deep(
                images
            )

            features.append(
                outputs.cpu().numpy()
            )

            labels.append(
                batch_labels.numpy()
            )

    features = np.concatenate(
        features,
        axis=0
    )

    labels = np.concatenate(
        labels,
        axis=0
    )

    return features, labels


# ------------------------------------------------------------
# 9. Extract TRAIN
# ------------------------------------------------------------

train_deep_features, train_deep_labels = (
    extract_deep_features(
        train_deep_loader,
        "TRAIN"
    )
)


# ------------------------------------------------------------
# 10. Extract VALIDATION
# ------------------------------------------------------------

val_deep_features, val_deep_labels = (
    extract_deep_features(
        val_deep_loader,
        "VALIDATION"
    )
)


# ------------------------------------------------------------
# 11. Save to disk
# ------------------------------------------------------------

TRAIN_DEEP_PATH = (
    "./train_deep_features_seed2026.npz"
)

VAL_DEEP_PATH = (
    "./val_deep_features_seed2026.npz"
)

np.savez_compressed(
    TRAIN_DEEP_PATH,
    features=train_deep_features,
    labels=train_deep_labels
)

np.savez_compressed(
    VAL_DEEP_PATH,
    features=val_deep_features,
    labels=val_deep_labels
)


# ------------------------------------------------------------
# 12. Verify alignment
# ------------------------------------------------------------

print("=" * 70)
print("DEEP FEATURES READY")
print("=" * 70)

print(
    "Train feature shape:",
    train_deep_features.shape
)

print(
    "Validation feature shape:",
    val_deep_features.shape
)

print(
    "Train labels shape:",
    train_deep_labels.shape
)

print(
    "Validation labels shape:",
    val_deep_labels.shape
)

print(
    "\nCheckpoint:",
    SAMPLER_CHECKPOINT
)

print(
    "Checkpoint Macro F1:",
    checkpoint_deep["macro_f1"]
)

print(
    "\nSaved:"
)

print(TRAIN_DEEP_PATH)
print(VAL_DEEP_PATH)

print("=" * 70)

In [ ]:
# ============================================================
# REBUILD ROI HANDCRAFTED FEATURES
# ============================================================

import os
import cv2
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from torchvision import datasets

from skimage.feature import (
    graycomatrix,
    graycoprops,
    local_binary_pattern
)


# ------------------------------------------------------------
# 1. Load datasets
# ------------------------------------------------------------

train_dataset_roi = datasets.ImageFolder(
    TRAIN_DIR
)

val_dataset_roi = datasets.ImageFolder(
    VAL_DIR
)


# ------------------------------------------------------------
# 2. ROI feature extraction
# ------------------------------------------------------------

def extract_roi_features(
    image_path,
    crop_ratio=0.80
):

    image = cv2.imread(
        image_path,
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        raise ValueError(
            f"Could not read image: {image_path}"
        )

    # Resize
    image = cv2.resize(
        image,
        (224, 224)
    )

    # --------------------------------------------------------
    # Center ROI
    # --------------------------------------------------------

    crop_size = int(
        224 * crop_ratio
    )

    start = (
        224 - crop_size
    ) // 2

    end = start + crop_size

    roi = image[
        start:end,
        start:end
    ]


    # --------------------------------------------------------
    # Intensity
    # --------------------------------------------------------

    features = {

        "roi_mean_intensity":
            np.mean(roi),

        "roi_std_intensity":
            np.std(roi),

        "roi_min_intensity":
            np.min(roi),

        "roi_max_intensity":
            np.max(roi),

        "roi_p10":
            np.percentile(roi, 10),

        "roi_p25":
            np.percentile(roi, 25),

        "roi_p50":
            np.percentile(roi, 50),

        "roi_p75":
            np.percentile(roi, 75),

        "roi_p90":
            np.percentile(roi, 90)
    }


    # --------------------------------------------------------
    # Entropy
    # --------------------------------------------------------

    histogram = cv2.calcHist(
        [roi],
        [0],
        None,
        [256],
        [0, 256]
    ).flatten()

    histogram = histogram / (
        histogram.sum() + 1e-8
    )

    features["roi_entropy"] = -np.sum(
        histogram *
        np.log2(histogram + 1e-10)
    )


    # --------------------------------------------------------
    # GLCM
    # --------------------------------------------------------

    roi_32 = (
        roi / 8
    ).astype(np.uint8)

    glcm = graycomatrix(
        roi_32,
        distances=[1, 2],
        angles=[
            0,
            np.pi / 4,
            np.pi / 2,
            3 * np.pi / 4
        ],
        levels=32,
        symmetric=True,
        normed=True
    )

    for prop in [
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation"
    ]:

        values = graycoprops(
            glcm,
            prop
        )

        features[
            f"roi_glcm_{prop}"
        ] = values.mean()


    # --------------------------------------------------------
    # LBP
    # --------------------------------------------------------

    lbp = local_binary_pattern(
        roi,
        P=8,
        R=1,
        method="uniform"
    )

    lbp_hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, 11),
        range=(0, 10)
    )

    lbp_hist = lbp_hist.astype(
        np.float32
    )

    lbp_hist /= (
        lbp_hist.sum() + 1e-8
    )

    for i, value in enumerate(lbp_hist):

        features[
            f"roi_lbp_{i}"
        ] = value

    return features


# ------------------------------------------------------------
# 3. Build DataFrame
# ------------------------------------------------------------

def build_roi_dataframe(
    dataset,
    split_name
):

    rows = []

    for image_path, label in tqdm(
        dataset.samples,
        desc=f"ROI features — {split_name}"
    ):

        features = extract_roi_features(
            image_path
        )

        features["label"] = label
        features["image_path"] = image_path

        rows.append(features)

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# 4. Extract
# ------------------------------------------------------------

train_roi_df = build_roi_dataframe(
    train_dataset_roi,
    "TRAIN"
)

val_roi_df = build_roi_dataframe(
    val_dataset_roi,
    "VALIDATION"
)


# ------------------------------------------------------------
# 5. Save immediately
# ------------------------------------------------------------

TRAIN_ROI_PATH = (
    "./train_roi_classical_features.csv"
)

VAL_ROI_PATH = (
    "./val_roi_classical_features.csv"
)

train_roi_df.to_csv(
    TRAIN_ROI_PATH,
    index=False
)

val_roi_df.to_csv(
    VAL_ROI_PATH,
    index=False
)


# ------------------------------------------------------------
# 6. Verify
# ------------------------------------------------------------

print("=" * 70)
print("ROI FEATURES REBUILT")
print("=" * 70)

print(
    "Train shape:",
    train_roi_df.shape
)

print(
    "Validation shape:",
    val_roi_df.shape
)

print(
    "Train file:",
    TRAIN_ROI_PATH
)

print(
    "Validation file:",
    VAL_ROI_PATH
)

print("=" * 70)

In [ ]:
# ============================================================
# HYBRID EXPERIMENT
# Step 2 — Extract 512-D Deep Features
# From Seed 2026 + Sampler Checkpoint
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from tqdm.auto import tqdm
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader


# ------------------------------------------------------------
# 1. Verify checkpoint exists
# ------------------------------------------------------------

CHECKPOINT_PATH = (
    "./best_resnet18_sampler_seed2026.pth"
)

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Checkpoint not found:\n{CHECKPOINT_PATH}\n"
        "Make sure the downloaded checkpoint was recreated "
        "or uploaded to this Colab runtime."
    )

print(
    "Checkpoint found:",
    CHECKPOINT_PATH
)


# ------------------------------------------------------------
# 2. Load ROI feature tables
# ------------------------------------------------------------

train_roi_df = pd.read_csv(
    "./train_roi_classical_features.csv"
)

val_roi_df = pd.read_csv(
    "./val_roi_classical_features.csv"
)


# ------------------------------------------------------------
# 3. Same preprocessing used by the CNN
# ------------------------------------------------------------

deep_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# 4. Dataset for deep feature extraction
# ------------------------------------------------------------

class DeepFeatureDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        image_path = self.df.loc[
            idx,
            "image_path"
        ]

        label = int(
            self.df.loc[
                idx,
                "label"
            ]
        )

        image = Image.open(
            image_path
        ).convert("L")

        image = self.transform(
            image
        )

        return image, label


# ------------------------------------------------------------
# 5. Build datasets and loaders
# ------------------------------------------------------------

train_deep_dataset = DeepFeatureDataset(
    train_roi_df,
    deep_transform
)

val_deep_dataset = DeepFeatureDataset(
    val_roi_df,
    deep_transform
)

train_deep_loader = DataLoader(
    train_deep_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

val_deep_loader = DataLoader(
    val_deep_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# 6. Build ResNet18
# ------------------------------------------------------------

model_deep = models.resnet18(
    weights=None
)

model_deep.fc = nn.Linear(
    model_deep.fc.in_features,
    5
)

model_deep = model_deep.to(device)


# ------------------------------------------------------------
# 7. Load trained CNN checkpoint
# ------------------------------------------------------------

checkpoint_deep = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model_deep.load_state_dict(
    checkpoint_deep["model_state_dict"]
)


# ------------------------------------------------------------
# 8. Replace classification head with Identity
# ------------------------------------------------------------

model_deep.fc = nn.Identity()

model_deep.eval()


# ------------------------------------------------------------
# 9. Extract deep features
# ------------------------------------------------------------

def extract_deep_features(
    loader,
    split_name
):

    features = []
    labels = []

    with torch.no_grad():

        for images, batch_labels in tqdm(
            loader,
            desc=f"Deep features — {split_name}"
        ):

            images = images.to(
                device,
                non_blocking=True
            )

            outputs = model_deep(
                images
            )

            features.append(
                outputs.cpu().numpy()
            )

            labels.append(
                batch_labels.numpy()
            )

    features = np.concatenate(
        features,
        axis=0
    )

    labels = np.concatenate(
        labels,
        axis=0
    )

    return features, labels


# ------------------------------------------------------------
# 10. Extract
# ------------------------------------------------------------

train_deep_features, train_deep_labels = (
    extract_deep_features(
        train_deep_loader,
        "TRAIN"
    )
)

val_deep_features, val_deep_labels = (
    extract_deep_features(
        val_deep_loader,
        "VALIDATION"
    )
)


# ------------------------------------------------------------
# 11. Alignment check
# ------------------------------------------------------------

assert len(train_deep_features) == len(
    train_roi_df
)

assert len(val_deep_features) == len(
    val_roi_df
)

assert np.array_equal(
    train_deep_labels,
    train_roi_df["label"].values
)

assert np.array_equal(
    val_deep_labels,
    val_roi_df["label"].values
)


# ------------------------------------------------------------
# 12. Save
# ------------------------------------------------------------

TRAIN_DEEP_PATH = (
    "./train_deep_features_seed2026.npz"
)

VAL_DEEP_PATH = (
    "./val_deep_features_seed2026.npz"
)

np.savez_compressed(
    TRAIN_DEEP_PATH,
    features=train_deep_features,
    labels=train_deep_labels
)

np.savez_compressed(
    VAL_DEEP_PATH,
    features=val_deep_features,
    labels=val_deep_labels
)


# ------------------------------------------------------------
# 13. Final report
# ------------------------------------------------------------

print("=" * 70)
print("DEEP FEATURE EXTRACTION COMPLETE")
print("=" * 70)

print(
    "Checkpoint Macro F1:",
    checkpoint_deep["macro_f1"]
)

print(
    "Train deep shape:",
    train_deep_features.shape
)

print(
    "Validation deep shape:",
    val_deep_features.shape
)

print(
    "\nSaved:"
)

print(TRAIN_DEEP_PATH)
print(VAL_DEEP_PATH)

print("=" * 70)

In [ ]:
# ============================================================
# HYBRID EXPERIMENT
# Deep-only vs Handcrafted-only vs Deep+Handcrafted
# Classifier: SVM-RBF
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    cohen_kappa_score
)


# ------------------------------------------------------------
# 1. Load deep features
# ------------------------------------------------------------

train_deep_data = np.load(
    "./train_deep_features_seed2026.npz"
)

val_deep_data = np.load(
    "./val_deep_features_seed2026.npz"
)

X_train_deep = train_deep_data["features"]
y_train = train_deep_data["labels"]

X_val_deep = val_deep_data["features"]
y_val = val_deep_data["labels"]


# ------------------------------------------------------------
# 2. Load handcrafted ROI features
# ------------------------------------------------------------

train_roi_df = pd.read_csv(
    "./train_roi_classical_features.csv"
)

val_roi_df = pd.read_csv(
    "./val_roi_classical_features.csv"
)

roi_feature_columns = [
    col
    for col in train_roi_df.columns
    if col not in [
        "label",
        "image_path"
    ]
]

X_train_roi = train_roi_df[
    roi_feature_columns
].values

X_val_roi = val_roi_df[
    roi_feature_columns
].values


# ------------------------------------------------------------
# 3. Alignment checks
# ------------------------------------------------------------

assert X_train_deep.shape[0] == X_train_roi.shape[0]
assert X_val_deep.shape[0] == X_val_roi.shape[0]

assert np.array_equal(
    y_train,
    train_roi_df["label"].values
)

assert np.array_equal(
    y_val,
    val_roi_df["label"].values
)


print("=" * 70)
print("FEATURE ALIGNMENT")
print("=" * 70)

print(
    "Deep train:",
    X_train_deep.shape
)

print(
    "ROI train:",
    X_train_roi.shape
)

print(
    "Deep validation:",
    X_val_deep.shape
)

print(
    "ROI validation:",
    X_val_roi.shape
)


# ------------------------------------------------------------
# 4. Create hybrid features
# ------------------------------------------------------------

X_train_hybrid = np.concatenate(
    [
        X_train_deep,
        X_train_roi
    ],
    axis=1
)

X_val_hybrid = np.concatenate(
    [
        X_val_deep,
        X_val_roi
    ],
    axis=1
)

print(
    "Hybrid train:",
    X_train_hybrid.shape
)

print(
    "Hybrid validation:",
    X_val_hybrid.shape
)


# ------------------------------------------------------------
# 5. Same SVM for all experiments
# ------------------------------------------------------------

def make_svm():

    return Pipeline([
        (
            "scaler",
            StandardScaler()
        ),

        (
            "svm",
            SVC(
                kernel="rbf",
                C=2.0,
                gamma="scale",
                class_weight="balanced"
            )
        )
    ])


# ------------------------------------------------------------
# 6. Evaluation helper
# ------------------------------------------------------------

def evaluate_model(
    model,
    X_train,
    X_val,
    name
):

    print("\n" + "=" * 70)
    print(f"TRAINING: {name}")
    print("=" * 70)

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_val
    )

    accuracy = accuracy_score(
        y_val,
        predictions
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_val,
            predictions
        )
    )

    macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    mae = mean_absolute_error(
        y_val,
        predictions
    )

    qwk = cohen_kappa_score(
        y_val,
        predictions,
        weights="quadratic"
    )

    print(
        f"Accuracy          : {accuracy:.4f}"
    )

    print(
        f"Balanced Accuracy : {balanced_accuracy:.4f}"
    )

    print(
        f"Macro F1          : {macro_f1:.4f}"
    )

    print(
        f"MAE               : {mae:.4f}"
    )

    print(
        f"QWK               : {qwk:.4f}"
    )

    return {
        "Model": name,
        "Accuracy": accuracy,
        "Balanced Accuracy": balanced_accuracy,
        "Macro F1": macro_f1,
        "MAE": mae,
        "QWK": qwk
    }


# ------------------------------------------------------------
# 7. Deep-only
# ------------------------------------------------------------

results = []

results.append(
    evaluate_model(
        make_svm(),
        X_train_deep,
        X_val_deep,
        "Deep-only SVM"
    )
)


# ------------------------------------------------------------
# 8. Handcrafted-only
# ------------------------------------------------------------

results.append(
    evaluate_model(
        make_svm(),
        X_train_roi,
        X_val_roi,
        "Handcrafted ROI-only SVM"
    )
)


# ------------------------------------------------------------
# 9. Hybrid
# ------------------------------------------------------------

results.append(
    evaluate_model(
        make_svm(),
        X_train_hybrid,
        X_val_hybrid,
        "Deep + Handcrafted Hybrid SVM"
    )
)


# ------------------------------------------------------------
# 10. Comparison
# ------------------------------------------------------------

results_df = pd.DataFrame(
    results
).sort_values(
    "Macro F1",
    ascending=False
)

print("\n" + "=" * 70)
print("FINAL FEATURE COMPARISON")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("=" * 70)

In [ ]:
import os
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

files_to_check = [
    "./best_resnet18_sampler_seed2026.pth",
    "./best_resnet18_D_seed2026.pth",
    "./train_roi_classical_features.csv",
    "./val_roi_classical_features.csv",
    "./train_deep_features_seed2026.npz",
    "./val_deep_features_seed2026.npz",
]

print("\nFile status:")
for path in files_to_check:
    print(
        os.path.basename(path),
        "->",
        "EXISTS" if os.path.exists(path) else "MISSING"
    )

In [ ]:
import os

path = "./best_resnet18_sampler_seed2026.pth"

print(
    "Checkpoint:",
    "FOUND" if os.path.exists(path) else "MISSING"
)

In [ ]:
# ============================================================
# ROI EXPERIMENT
# Step 1 — Preview Knee-Centered Crop
# ============================================================

import os
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import datasets


# ------------------------------------------------------------
# 1. Rebuild dataset if necessary
# ------------------------------------------------------------

train_dataset_roi_preview = datasets.ImageFolder(
    TRAIN_DIR
)

# ------------------------------------------------------------
# 2. ROI crop function
# ------------------------------------------------------------

def get_center_roi(
    image,
    crop_ratio=0.70
):
    """
    Take a centered crop from the image.

    crop_ratio=0.70 means 70% of width and height
    are retained.
    """

    h, w = image.shape[:2]

    crop_h = int(h * crop_ratio)
    crop_w = int(w * crop_ratio)

    y1 = (h - crop_h) // 2
    x1 = (w - crop_w) // 2

    y2 = y1 + crop_h
    x2 = x1 + crop_w

    return image[
        y1:y2,
        x1:x2
    ]


# ------------------------------------------------------------
# 3. Select random images
# ------------------------------------------------------------

random.seed(2026)

sample_indices = random.sample(
    range(len(train_dataset_roi_preview)),
    12
)


# ------------------------------------------------------------
# 4. Display original + ROI
# ------------------------------------------------------------

fig, axes = plt.subplots(
    12,
    2,
    figsize=(8, 36)
)

for row, idx in enumerate(sample_indices):

    image_path, label = (
        train_dataset_roi_preview.samples[idx]
    )

    image = cv2.imread(
        image_path,
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        continue

    roi = get_center_roi(
        image,
        crop_ratio=0.70
    )

    # Resize only for display
    original_display = cv2.resize(
        image,
        (224, 224)
    )

    roi_display = cv2.resize(
        roi,
        (224, 224)
    )

    axes[row, 0].imshow(
        original_display,
        cmap="gray"
    )

    axes[row, 0].set_title(
        f"Original | KL{label}"
    )

    axes[row, 0].axis("off")

    axes[row, 1].imshow(
        roi_display,
        cmap="gray"
    )

    axes[row, 1].set_title(
        "Center ROI (70%)"
    )

    axes[row, 1].axis("off")


plt.tight_layout()
plt.show()

print("=" * 70)
print("ROI PREVIEW COMPLETE")
print("=" * 70)
print("Crop ratio: 70%")
print("Images shown: 12")
print("=" * 70)